# SELL-based Resource-Aware Neuro-Symbolic KBQA

This notebook is an experimental scaffold for:

**Neuro-Symbolic KG/KB-QA + resource-aware reasoning via SELL**

It runs offline on a tiny in-memory KB, but keeps interfaces you can later swap to a real KB
(e.g., Freebase via a SPARQL endpoint).

Included:
- Pipeline skeleton (linking -> retrieval -> candidates -> SELL check -> execution -> scoring)
- One GrailQA-derived example (from KBQA-Agent, source=grailqa)
- One ComplexWebQuestions (CWQ) example from the NAACL 2018 paper figure


## Embedded examples

We store just *one* record per dataset as small Python dicts (not full dataset dumps).


In [ ]:
# --- Minimal example records (one from GrailQA source, one from CWQ paper figure) ---

GRAILQA_EXAMPLE = {
    "qid": "4300563004000_grailqa",
    "source": "grailqa",
    "question": (
        "what position did pat connaughton, author of did you see that thing? "
        "that's sidat-singh! the syracuse walking dream!, play?"
    ),
    # Keep only the entities used by the gold logical form.
    "entities": {
        "Did you see that thing? That's Sidat-Singh! The Syracuse Walking Dream!": "m.09rl290",
        "Pat Connaughton": "m.0j50kb6",
    },
    # Gold GrailQA S-expression (as displayed in KBQA-Agent viewer)
    "s_expression": (
        "(AND basketball.basketball_position "
        "(AND "
        "(JOIN basketball.basketball_position.players "
        "(JOIN (R media_common.quotation.author) m.09rl290)) "
        "(JOIN basketball.basketball_position.players m.0j50kb6)))"
    ),
    "answer": [
        {"answer_argument": "m.05ch8k4", "answer_type": "Entity", "entity_name": "Guard"}
    ],
    # Ground-truth "agent tool" actions (KBQA-Agent provides these)
    "actions": [
        "get_relations(m.09rl290)",
        "get_neighbors(m.09rl290,media_common.quotation.author)",
        "get_relations(#0)",
        "get_neighbors(#0,basketball.basketball_player.position_s)",
        "get_relations(m.0j50kb6)",
        "get_neighbors(m.0j50kb6,basketball.basketball_player.position_s)",
        "intersection(#1,#2)",
    ],
}

# CWQ example from the original NAACL 2018 paper (Figure 3: seed -> SPARQL -> machine q -> NL q).
CWQ_EXAMPLE = {
    "id": "cwq_fig3_demo",
    "source": "complexwebquestions",
    "seed_question": "What movies have Robert Pattinson starred in?",
    "natural_question": "Which Robert Pattinson film was produced by Erwin Stoff?",
    "sparql_triples": [
        # Represented as (subject, predicate, object), with variables prefixed by '?'.
        ("ns:robert_pattinson", "ns:film.actor.film", "?c"),
        ("?c", "ns:film.performance.film", "?x"),
        ("?x", "ns:film.film.produced_by", "ns:erwin_stoff"),
    ],
}

print("Loaded examples:")
print(" - GrailQA qid:", GRAILQA_EXAMPLE["qid"])
print(" - CWQ id:", CWQ_EXAMPLE["id"])


## Step 0 - A tiny KB that runs offline

We create a tiny in-memory KB with only the triples needed for the two examples.

When you move to real experiments, replace this with a SPARQL adapter but keep the same
neighbors/reverse_neighbors interface.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple

Entity = str
Relation = str
Triple = Tuple[Entity, Relation, Entity]

@dataclass
class ToyKB:
    triples: List[Triple]

    def neighbors(self, s: Entity, r: Relation) -> Set[Entity]:
        return {o for (ss, rr, o) in self.triples if ss == s and rr == r}

    def reverse_neighbors(self, o: Entity, r: Relation) -> Set[Entity]:
        return {s for (s, rr, oo) in self.triples if oo == o and rr == r}

# --- Toy KB for the GrailQA example ---
TOY_KB_GRAILQA = ToyKB(triples=[
    # Make (JOIN (R media_common.quotation.author) m.09rl290) return Pat Connaughton
    ("m.0j50kb6", "media_common.quotation.author", "m.09rl290"),

    # Make positions of Pat include Guard
    ("m.0j50kb6", "basketball.basketball_position.players", "m.05ch8k4"),
])

# --- Toy KB for the CWQ example ---
TOY_KB_CWQ = ToyKB(triples=[
    ("ns:robert_pattinson", "ns:film.actor.film", "ns:perf_demo_1"),
    ("ns:robert_pattinson", "ns:film.actor.film", "ns:perf_demo_2"),
    ("ns:perf_demo_1", "ns:film.performance.film", "ns:film_demo_1"),
    ("ns:perf_demo_2", "ns:film.performance.film", "ns:film_demo_2"),
    ("ns:film_demo_1", "ns:film.film.produced_by", "ns:erwin_stoff"),
    ("ns:film_demo_2", "ns:film.film.produced_by", "ns:someone_else"),
])

print("Toy KBs ready:", len(TOY_KB_GRAILQA.triples), len(TOY_KB_CWQ.triples))


## Step 1 - Execute a minimal GrailQA S-expression subset

Supports JOIN, reverse JOIN (R), AND (intersection). Enough for the embedded example.


In [ ]:
def tokenize_sexpr(s: str):
    s = s.replace("(", " ( ").replace(")", " ) ")
    return [t for t in s.split() if t]

def parse_sexpr(tokens):
    tok = tokens.pop(0)
    if tok == "(":
        out = []
        while tokens[0] != ")":
            out.append(parse_sexpr(tokens))
        tokens.pop(0)
        return out
    if tok == ")":
        raise ValueError("Unexpected ')'")
    return tok

def eval_sexpr(ast, kb: ToyKB):
    if isinstance(ast, str):
        return {ast}

    head = ast[0]
    if head == "JOIN":
        rel = ast[1]
        child = ast[2]
        base = eval_sexpr(child, kb)

        if isinstance(rel, list) and rel and rel[0] == "R":
            rname = rel[1]
            out = set()
            for o in base:
                out |= kb.reverse_neighbors(o, rname)
            return out

        if isinstance(rel, str):
            out = set()
            for s in base:
                out |= kb.neighbors(s, rel)
            return out

        raise ValueError(f"Bad rel node: {rel}")

    if head == "AND":
        # (AND type e1 e2 ...)
        exprs = ast[2:]
        res = eval_sexpr(exprs[0], kb)
        for e in exprs[1:]:
            res &= eval_sexpr(e, kb)
        return res

    raise ValueError(f"Unsupported operator: {head}")

tokens = tokenize_sexpr(GRAILQA_EXAMPLE["s_expression"])
ast = parse_sexpr(tokens)
pred = eval_sexpr(ast, TOY_KB_GRAILQA)
pred


## Step 2 - Execute CWQ triple patterns (mini SPARQL)

A tiny conjunctive-query evaluator for lists of (s, p, o) where variables start with '?'.


In [ ]:
def execute_triple_patterns(triples, kb: ToyKB):
    def is_var(x: str) -> bool:
        return x.startswith("?")

    sols = [dict()]
    for (s, p, o) in triples:
        new_sols = []
        for sol in sols:
            for (ss, rr, oo) in kb.triples:
                if rr != p:
                    continue

                # unify subject
                if is_var(s):
                    if s in sol and sol[s] != ss:
                        continue
                else:
                    if s != ss:
                        continue

                # unify object
                if is_var(o):
                    if o in sol and sol[o] != oo:
                        continue
                else:
                    if o != oo:
                        continue

                sol2 = dict(sol)
                if is_var(s):
                    sol2[s] = ss
                if is_var(o):
                    sol2[o] = oo
                new_sols.append(sol2)
        sols = new_sols
    return sols

sols = execute_triple_patterns(CWQ_EXAMPLE["sparql_triples"], TOY_KB_CWQ)
answers = sorted({sol["?x"] for sol in sols if "?x" in sol})
sols[:3], answers


## Step 3 - SELL placeholder: resource-aware admissibility

This is the hook where SELL proof search/derivation lives.

Executable placeholder:
- count JOIN nodes as 1 hop
- accept candidate if hop_count <= max_hops


In [ ]:
from dataclasses import dataclass

def hop_count_sexpr(ast):
    if isinstance(ast, str):
        return 0
    if ast[0] == "JOIN":
        return 1 + hop_count_sexpr(ast[2])
    if ast[0] == "AND":
        return sum(hop_count_sexpr(e) for e in ast[2:])
    return 0

@dataclass(frozen=True)
class ResourcePolicy:
    max_hops: int = 3

def sell_resource_check(candidate_ast, policy: ResourcePolicy) -> bool:
    return hop_count_sexpr(candidate_ast) <= policy.max_hops

policy = ResourcePolicy(max_hops=3)
print("Hop count:", hop_count_sexpr(ast))
print("Admissible?", sell_resource_check(ast, policy))


## Step 4 - Pipeline skeleton (interfaces)

Modules:
1) Entity linking
2) Candidate generation
3) SELL check
4) Execution
5) Ranking


In [ ]:
from typing import Any

class EntityLinker:
    def link(self, question: str) -> Dict[str, str]:
        raise NotImplementedError

class HeuristicEntityLinker(EntityLinker):
    def __init__(self, known_entities: Dict[str, str]):
        self.known_entities = known_entities

    def link(self, question: str) -> Dict[str, str]:
        q = question.lower()
        return {name: eid for name, eid in self.known_entities.items() if name.lower() in q}

class CandidateGenerator:
    def generate(self, question: str, entities: Dict[str, str]) -> List[Dict[str, Any]]:
        raise NotImplementedError

class GoldOnlyCandidateGenerator(CandidateGenerator):
    def __init__(self, gold_record: Dict[str, Any]):
        self.gold = gold_record

    def generate(self, question: str, entities: Dict[str, str]) -> List[Dict[str, Any]]:
        return [{"kind": "sexpr", "program": self.gold["s_expression"], "meta": {"source": "gold"}}]

class Executor:
    def execute(self, candidate: Dict[str, Any]) -> Set[str]:
        raise NotImplementedError

class SExprExecutor(Executor):
    def __init__(self, kb: ToyKB):
        self.kb = kb

    def execute(self, candidate: Dict[str, Any]) -> Set[str]:
        cand_ast = parse_sexpr(tokenize_sexpr(candidate["program"]))
        return eval_sexpr(cand_ast, self.kb)

class SimpleRanker:
    def rank(self, candidates: List[Dict[str, Any]], answers: List[Set[str]]) -> int:
        return 0

def run_pipeline_one(question: str, linker, generator, executor, policy, ranker):
    ents = linker.link(question)
    candidates = generator.generate(question, ents)

    admissible = []
    answers = []
    for c in candidates:
        if c["kind"] == "sexpr":
            cand_ast = parse_sexpr(tokenize_sexpr(c["program"]))
            if not sell_resource_check(cand_ast, policy):
                continue
        admissible.append(c)
        answers.append(executor.execute(c))

    if not admissible:
        return None

    best = ranker.rank(admissible, answers)
    return {"entities": ents, "candidate": admissible[best], "answers": answers[best]}

linker = HeuristicEntityLinker(GRAILQA_EXAMPLE["entities"])
generator = GoldOnlyCandidateGenerator(GRAILQA_EXAMPLE)
executor = SExprExecutor(TOY_KB_GRAILQA)
ranker = SimpleRanker()

out = run_pipeline_one(
    question=GRAILQA_EXAMPLE["question"],
    linker=linker,
    generator=generator,
    executor=executor,
    policy=ResourcePolicy(max_hops=3),
    ranker=ranker,
)
out


## Step 5 - Metrics (EM/F1 on answer sets)

In [ ]:
def f1_set(pred: Set[str], gold: Set[str]) -> float:
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    inter = len(pred & gold)
    p = inter / len(pred)
    r = inter / len(gold)
    return 2 * p * r / (p + r) if (p + r) else 0.0

gold = {a["answer_argument"] for a in GRAILQA_EXAMPLE["answer"]}
pred = out["answers"] if out else set()

print("Gold:", gold)
print("Pred:", pred)
print("EM:", float(pred == gold))
print("F1:", f1_set(pred, gold))


## Optional - LLM/Codex hook (stub)

Replace GoldOnlyCandidateGenerator with an LLM-assisted generator + reranker.


In [ ]:
def llm_propose_candidates_stub(question: str, entities: Dict[str, str], k: int = 5):
    # Return k candidate programs (S-expressions / SPARQL).
    # Implement this with your Codex/LLM call.
    # Recommended: structured JSON output {program, rationale, confidence}.
    raise NotImplementedError
